In [ ]:
import os
import shutil
from sklearn.model_selection import train_test_split


In [ ]:
for folder in ['train', 'test', 'valid']:
    for subfolder in ['images', 'labels']:
        os.makedirs(f'../data/{folder}/{subfolder}', exist_ok=True)

In [ ]:
images_folder_path = '../annotated_data/train/images'
labeles_folder_path = '../annotated_data/train/labels'

images = os.listdir(images_folder_path)
labels = os.listdir(labeles_folder_path)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test, y_test, test_size=0.5, random_state=42
)

In [ ]:
def move_files(list, folder_name = 'train', files_type = 'images'):
    for file in list:
        shutil.move(f'../annotated_data/train/{files_type}/{file}', f'../data/{folder_name}/{files_type}/{file}')

move_files(X_train, 'train')
move_files(y_train, 'train', 'labels')

move_files(X_test, 'test')
move_files(y_test, 'test', 'labels')

move_files(X_val, 'valid')
move_files(y_val, 'valid', 'labels')

In [ ]:
for folder in ['train','test', 'valid']:
        images_folder_path = f'../data/{folder}/images'
        labeles_folder_path = f'../data/{folder}/labels'

        print(len(os.listdir(images_folder_path)))
        print(len(os.listdir(labeles_folder_path)))

In [ ]:
import albumentations as A
import cv2
import numpy as np

In [ ]:
def read_yolo_labels(label_path):
    bboxes, labels = [], []
    with open(label_path) as f:
        for line in f:
            cls, cx, cy, w, h = map(float, line.strip().split())
            bboxes.append([cx, cy, w, h])
            labels.append(int(cls))
    return bboxes, labels

In [ ]:
def save_yolo_labels(label_path, bboxes, labels):
    with open(label_path, "w") as f:
        for bbox, cls in zip(bboxes, labels):
            x, y, w, h = bbox
            f.write(f"{cls} {x} {y} {w} {h}\n")

In [ ]:
pipeline = A.Compose([
    A.Affine(rotate=[-45, 45], p=0.5),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.RandomCrop(width=400, height=400, p=0.3), 
], bbox_params=A.BboxParams(
    coord_format='yolo',
    label_fields=['labels'],
    clip_bboxes_on_input=True,
    clip_after_transform=True,
    filter_invalid_bboxes=True,
))

In [ ]:
images_folder = '../data/train/images'
labels_folder = '../data/train/labels'

images_list = list(os.listdir(images_folder))

for image_file in images_list:
    image_path = os.path.join(images_folder, image_file)

    label_file = os.path.splitext(image_file)[0] + ".txt"
    label_path = os.path.join(labels_folder, label_file)

    if not os.path.exists(label_path):
        continue 

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    bboxes, labels = read_yolo_labels(label_path=label_path)

    for i in range(4):
        transformed = pipeline(image=image, bboxes=bboxes, labels=labels)
        
        if len(transformed["bboxes"]) == 0:
            continue

        save_yolo_labels(f'{labels_folder}/{i}_{label_file}', 
                         bboxes=transformed["bboxes"], 
                         labels=transformed["labels"])
        cv2.imwrite(f'{images_folder}/{i}_{image_file}', 
                    cv2.cvtColor(transformed["image"], cv2.COLOR_RGB2BGR))
        

In [ ]:
images_folder = '../data/train/images'
labels_folder = '../data/train/labels'

images_list = list(os.listdir(images_folder))
labels_list = list(os.listdir(labels_folder))


In [ ]:
import random
import matplotlib.pyplot as plt
random_imgs = random.sample(images_list, k = 9)

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(10, 10))
for i, image_file in enumerate(random_imgs):
    image_path = os.path.join(images_folder, image_file)

    label_file = os.path.splitext(image_file)[0] + ".txt"
    label_path = os.path.join(labels_folder, label_file)

    if not os.path.exists(label_path):
        continue 

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    bboxes, labels = read_yolo_labels(label_path=label_path)
    color = (0, 255, 0)  # Green
    thickness = 3
    img_width, img_height = image.shape[1], image.shape[0]

    for bbox in bboxes:
        cx, cy, w, h = bbox
        x1 = (cx - w/2) * img_width
        y1 = (cy - h/2) * img_height
        x2 = (cx + w/2) * img_width
        y2 = (cy + h/2) * img_height
        cv2.rectangle(image, (int(x1), int(y1)), (int(x2), int(y2)), color, thickness)
    
    axes[i//3, i%3].imshow(image)

plt.tight_layout() 
plt.show()